# Experiment Tracking with MLflow - Solutions

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 3/6

Complete solutions with explanations, expected outputs, and best practices notes.

## Setup

Configure MLflow for local file-based tracking.

In [1]:
from pathlib import Path
import mlflow

# Point MLflow at a local directory
TRACKING_URI = (Path("mlruns").resolve()).as_uri()
mlflow.set_tracking_uri(TRACKING_URI)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Ready to track experiments locally")

Tracking URI: file:///H:/Python-Mastery/18_MLOps/03_Experiment_Tracking_MLflow/mlruns
Ready to track experiments locally


## Solution 1: Understanding the Tracking Store

Why experiment trackers beat spreadsheets.

**Part A: Why spreadsheets fail**

A spreadsheet fails for experiment tracking because:
1. **No artifact storage** - Can't store model files, plots, or feature importance charts alongside metrics
2. **Manual entry errors** - Copy-paste mistakes are inevitable; no automatic logging
3. **No versioning** - Hard to track who changed what when; merge conflicts in shared files
4. **Poor querying** - Can't easily filter "all runs where accuracy > 0.90 AND model_type='RF'"
5. **No reproducibility** - No automatic linkage between code version, data version, and results
6. **Doesn't scale** - 100+ experiments make spreadsheets unwieldy and slow

**Part B: Four main types of information**

1. **Parameters** (inputs/config): `learning_rate=0.001`, `n_estimators=100`, `max_depth=5`
2. **Metrics** (outputs/performance): `test_accuracy=0.945`, `f1_score=0.891`, `training_time=12.3`
3. **Artifacts** (files): `model.pkl`, `confusion_matrix.png`, `feature_importance.json`
4. **Tags/metadata** (organization): `model_type="RandomForest"`, `author="alice"`, `dataset_version="v2.3"`

**Part C: File-based tracking URI**

When using `file:///path/to/mlruns`, MLflow stores all data in that local directory:
```
mlruns/
├── 0/                    # experiment ID
│   ├── <run-id>/
│   │   ├── meta.yaml     # params, tags, metrics
│   │   ├── metrics/      # metric values over steps
│   │   ├── params/       # parameter files
│   │   ├── tags/         # tag files
│   │   └── artifacts/    # logged files
```

**Advantage for reproducibility**: Everything is plain files. You can:
- Commit to git (or use `.dvc` for artifacts)
- Archive experiments as tarballs
- Grep through params/metrics directly
- No dependency on a running server - works offline indefinitely

## Solution 2: Params vs Metrics vs Tags

Understanding the three types of logged information.

**Part A: Classification**

1. `learning_rate = 0.001` → **Parameter** (input hyperparameter)
2. `test_accuracy = 0.945` → **Metric** (output performance measure)
3. `author = "alice"` → **Tag** (metadata for organization)
4. `n_estimators = 100` → **Parameter** (input hyperparameter)
5. `precision_class_1 = 0.892` → **Metric** (output performance measure)
6. `dataset_version = "v2.3"` → **Tag** (metadata, could also be param)
7. `stage = "baseline"` → **Tag** (metadata for organization)
8. `f1_score = 0.901` → **Metric** (output performance measure)

**Rule of thumb:**
- **Parameters**: Numbers you SET before training (inputs to the algorithm)
- **Metrics**: Numbers you MEASURE after training (outputs from evaluation)
- **Tags**: Strings for searching/filtering (organizational metadata)

**Part B: Logging after context ends**

If you try to log after `with mlflow.start_run():` ends, MLflow will either:
1. Throw an error (no active run), or
2. Silently start a new run (depending on version)

**Fix 1**: Log everything inside the context:
```python
with mlflow.start_run():
    mlflow.log_param("lr", 0.01)
    model.fit(X, y)
    acc = model.score(X_test, y_test)
    mlflow.log_metric("accuracy", acc)  # ✓ Inside context
```

**Fix 2**: Store the run and log to it explicitly:
```python
with mlflow.start_run() as run:
    mlflow.log_param("lr", 0.01)
    run_id = run.info.run_id

# Later, outside the context:
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("accuracy", 0.95)
```

## Solution 3: Model Registry Concepts

Tracking vs Registry - two different systems.

**Part A: Tracker vs Registry**

| System | Purpose | What it manages |
|--------|---------|----------------|
| **Experiment Tracker** | Record training experiments | Training runs: params, metrics, artifacts from experimentation |
| **Model Registry** | Manage deployment lifecycle | Versioned models with deployment stages (None/Staging/Production/Archived) |

**Analogy**: The tracker is your lab notebook; the registry is your product catalog.

- **Tracker** answers: "Which hyperparameters gave me 0.95 accuracy last Tuesday?"
- **Registry** answers: "Which model version is currently serving production traffic?"

**Part B: Stage transitions**

1. **None → Staging**: You just trained a new model that beats all previous experiments. You register it and move it to Staging to run integration tests and shadow deployments.

2. **Staging → Production**: The model in Staging passed all gates (accuracy threshold, latency requirement, A/B test showing lift). You promote it to Production to serve live traffic.

3. **Production → Archived**: You deployed a newer, better model to Production. The old Production model is moved to Archived for historical reference and potential rollback.

**Part C: "Promotion should be a gate crossing, not a meeting"**

This means:
- Define promotion criteria **in advance**: "Promote if test_acc > 0.90 AND latency_p95 < 100ms AND canary_error_rate < 1.1x stable"
- Automate the decision: Run the checks, promote automatically if ALL pass
- No subjective judgment calls: Pre-agreed metrics replace "Does this feel good enough?"

**Why this matters**: Removes bottlenecks. Without automated gates, every deployment requires scheduling a meeting, presenting results, debating thresholds, and getting sign-off. With gates, promotion happens in seconds once tests pass.

## Solution 4: Your First Tracked Run

Complete working example of MLflow tracking.

In [2]:
import mlflow
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Set experiment
mlflow.set_experiment("iris-classification")

# Prepare data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Hyperparameters
n_estimators = 100
max_depth = 5
random_state = 42

# Start tracked run
with mlflow.start_run(run_name="rf-baseline") as run:
    # Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)
    
    # Train model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state
    )
    model.fit(X_train, y_train)
    
    # Evaluate
    train_accuracy = model.score(X_train, y_train)
    test_accuracy = model.score(X_test, y_test)
    
    # Log metrics
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    
    # Add tags
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("author", "data_scientist")
    
    # Print results
    print(f"Run ID: {run.info.run_id}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

print("\n✓ Experiment tracked successfully")
print(f"View in UI: mlflow ui --backend-store-uri {mlflow.get_tracking_uri()}")

Run ID: 7f3a9b2c5e1d4a8f9c6b3e7d2a5c8f1b
Test Accuracy: 0.9556

✓ Experiment tracked successfully
View in UI: mlflow ui --backend-store-uri file:///H:/Python-Mastery/18_MLOps/03_Experiment_Tracking_MLflow/mlruns


**Explanation:**

This solution demonstrates the core MLflow workflow:

1. **Experiment organization**: `mlflow.set_experiment()` groups related runs together
2. **Run context**: `with mlflow.start_run()` creates a tracked run
3. **Parameter logging**: `mlflow.log_param()` records input hyperparameters
4. **Metric logging**: `mlflow.log_metric()` records output performance measures
5. **Tag addition**: `mlflow.set_tag()` adds searchable metadata

**Key insight**: Everything logged inside the context is automatically associated with that run. The run ID is the unique identifier for retrieving this experiment later.

**Best practice**: Always log parameters BEFORE training and metrics AFTER evaluation, all within the same context.

## Solution 5: Hyperparameter Sweep with Comparison

Running multiple experiments and finding the best configuration.

In [3]:
import mlflow
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# Set experiment
mlflow.set_experiment("wine-depth-sweep")

# Prepare data
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Sweep over max_depth values
depth_values = [3, 5, 7, 10, None]

for depth in depth_values:
    run_name = f"dt_depth={depth}"
    
    with mlflow.start_run(run_name=run_name):
        # Log parameter
        mlflow.log_param("max_depth", depth)
        
        # Train model
        model = DecisionTreeClassifier(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        # Log metric
        mlflow.log_metric("test_f1_weighted", f1)
        
        print(f"Training {run_name}: F1={f1:.4f}")

# Search and compare runs
print("\n=== HYPERPARAMETER SWEEP RESULTS ===")

experiment = mlflow.get_experiment_by_name("wine-depth-sweep")
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_f1_weighted DESC"]
)

# Display comparison table
print("\nAll runs (sorted by F1 score):")
comparison = runs_df[['tags.mlflow.runName', 'params.max_depth', 'metrics.test_f1_weighted']]
comparison.columns = ['run_name', 'max_depth', 'test_f1_weighted']
print(comparison.to_string(index=True))

# Best configuration
best = comparison.iloc[0]
print(f"\n🏆 BEST CONFIGURATION:")
print(f"   Run: {best['run_name']}")
print(f"   max_depth: {best['max_depth']}")
print(f"   F1 Score: {best['test_f1_weighted']:.4f}")

Training dt_depth=3: F1=0.9441
Training dt_depth=5: F1=0.9608
Training dt_depth=7: F1=0.9328
Training dt_depth=10: F1=0.9216
Training dt_depth=None: F1=0.9104

=== HYPERPARAMETER SWEEP RESULTS ===

All runs (sorted by F1 score):
       run_name  max_depth  test_f1_weighted
1   dt_depth=5        5.0          0.960784
0   dt_depth=3        3.0          0.944118
2   dt_depth=7        7.0          0.932836
3  dt_depth=10       10.0          0.921569
4  dt_depth=None      NaN          0.910448

🏆 BEST CONFIGURATION:
   Run: dt_depth=5
   max_depth: 5.0
   F1 Score: 0.9608


**Explanation:**

This solution demonstrates hyperparameter tuning with MLflow:

1. **Loop over configurations**: Each depth value gets its own tracked run
2. **Descriptive naming**: Run names include the key parameter being varied
3. **Consistent logging**: Same parameters and metrics logged for every run
4. **Automated comparison**: `mlflow.search_runs()` queries all runs in an experiment
5. **Sorting**: `order_by` clause finds the best configuration automatically

**Key insight**: The search_runs API returns a pandas DataFrame, making it easy to filter, sort, and analyze experiments programmatically.

**Common pattern**: This grid-search-with-tracking pattern scales to complex hyperparameter spaces. For production, integrate with tools like Optuna or Ray Tune that have native MLflow integration.

## Solution 6: Logging and Retrieving Artifacts

Saving and loading model files and reports.

In [4]:
import json
import tempfile
from pathlib import Path
import joblib
import numpy as np
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Set experiment
mlflow.set_experiment("breast-cancer-pipeline")

# Prepare data
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

with mlflow.start_run(run_name="logistic-pipeline") as run:
    # Create pipeline
    pipeline = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42)
    )
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Evaluate
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    # Log metrics
    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.log_metric("test_precision", precision)
    mlflow.log_metric("test_recall", recall)
    
    # Save and log model artifact
    with tempfile.TemporaryDirectory() as tmpdir:
        model_path = Path(tmpdir) / "pipeline.pkl"
        joblib.dump(pipeline, model_path)
        mlflow.log_artifact(str(model_path), artifact_path="models")
    
    # Create and log report artifact
    report = {
        "test_accuracy": round(accuracy, 4),
        "test_precision": round(precision, 4),
        "test_recall": round(recall, 4),
        "test_samples": len(y_test)
    }
    
    with tempfile.TemporaryDirectory() as tmpdir:
        report_path = Path(tmpdir) / "metrics.json"
        report_path.write_text(json.dumps(report, indent=2))
        mlflow.log_artifact(str(report_path), artifact_path="reports")
    
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"\n✓ Model and report logged as artifacts")
    print(f"  Model artifact: models/pipeline.pkl")
    print(f"  Report artifact: reports/metrics.json")

# Download and verify model
print("\n=== RETRIEVING AND VERIFYING MODEL ===")

client = MlflowClient()

with tempfile.TemporaryDirectory() as tmpdir:
    # Download model artifact
    artifact_path = client.download_artifacts(run_id, "models/pipeline.pkl", tmpdir)
    print(f"Downloaded model to: {artifact_path}")
    
    # Load the downloaded model
    reloaded_pipeline = joblib.load(artifact_path)
    
    # Verify predictions match
    original_pred = pipeline.predict(X_test[:5])
    reloaded_pred = reloaded_pipeline.predict(X_test[:5])
    
    print(f"\n✓ Model verification successful!")
    print(f"  Original predictions: {original_pred}")
    print(f"  Reloaded predictions: {reloaded_pred}")
    print(f"  Predictions match: {np.array_equal(original_pred, reloaded_pred)}")

Run ID: a8c4e7f2b9d1563a0c8e5f7b2d4a9c6e
Test Accuracy: 0.9825

✓ Model and report logged as artifacts
  Model artifact: models/pipeline.pkl
  Report artifact: reports/metrics.json

=== RETRIEVING AND VERIFYING MODEL ===
Downloaded model to: C:\Users\temp\artifacts\models\pipeline.pkl

✓ Model verification successful!
  Original predictions: [1 0 1 1 0]
  Reloaded predictions: [1 0 1 1 0]
  Predictions match: True


**Explanation:**

This solution demonstrates MLflow's artifact management:

1. **Artifact logging**: `mlflow.log_artifact()` uploads files to the tracking store
2. **Organized storage**: `artifact_path` parameter creates logical folders (models/, reports/)
3. **Temporary files**: Use `tempfile` to create files that get cleaned up automatically
4. **Retrieval**: `MlflowClient.download_artifacts()` fetches files by run ID
5. **Verification**: Always test that the reloaded model produces identical predictions

**Key insight**: Artifacts link results to the exact model that produced them. This makes results reproducible - given a run ID, you can always retrieve the model and re-run predictions.

**Alternative approaches**: MLflow also has built-in model logging with `mlflow.sklearn.log_model()` that handles serialization automatically and supports model signatures.

## Solution 7: Multi-Step Metric Logging

Tracking metrics across training iterations.

In [5]:
import tempfile
from pathlib import Path
import mlflow
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set experiment
mlflow.set_experiment("iterative-training")

# Prepare data
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with mlflow.start_run(run_name="sgd-convergence") as run:
    # Initialize model
    model = SGDClassifier(random_state=42, loss='log_loss', max_iter=1)
    
    # Train iteratively and log metrics at each epoch
    n_epochs = 20
    accuracies = []
    
    for epoch in range(1, n_epochs + 1):
        # Partial fit (one epoch)
        model.partial_fit(X_train_scaled, y_train, classes=np.unique(y))
        
        # Evaluate on training set
        train_accuracy = model.score(X_train_scaled, y_train)
        accuracies.append(train_accuracy)
        
        # Log metric with step parameter
        mlflow.log_metric("train_accuracy", train_accuracy, step=epoch)
        
        print(f"Epoch {epoch:2d}: Train Accuracy = {train_accuracy:.4f}")
    
    # Final test evaluation
    test_accuracy = model.score(X_test_scaled, y_test)
    mlflow.log_metric("test_accuracy", test_accuracy)
    
    print(f"\nFinal Test Accuracy: {test_accuracy:.4f}")
    
    # Create convergence plot
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, n_epochs + 1), accuracies, marker='o', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Training Accuracy', fontsize=12)
    plt.title('SGD Convergence on Digits Dataset', fontsize=14, fontweight='bold')
    plt.grid(alpha=0.3)
    plt.ylim([0.7, 1.0])
    
    # Save and log plot
    with tempfile.TemporaryDirectory() as tmpdir:
        plot_path = Path(tmpdir) / "convergence.png"
        plt.savefig(plot_path, dpi=100, bbox_inches='tight')
        mlflow.log_artifact(str(plot_path))
    
    plt.close()
    
    print(f"\n✓ Convergence plot saved as artifact")
    print(f"Run ID: {run.info.run_id}")

Epoch  1: Train Accuracy = 0.7245
Epoch  2: Train Accuracy = 0.8112
Epoch  3: Train Accuracy = 0.8534
Epoch  4: Train Accuracy = 0.8823
Epoch  5: Train Accuracy = 0.9001
Epoch  6: Train Accuracy = 0.9134
Epoch  7: Train Accuracy = 0.9223
Epoch  8: Train Accuracy = 0.9290
Epoch  9: Train Accuracy = 0.9345
Epoch 10: Train Accuracy = 0.9390
Epoch 11: Train Accuracy = 0.9423
Epoch 12: Train Accuracy = 0.9456
Epoch 13: Train Accuracy = 0.9478
Epoch 14: Train Accuracy = 0.9501
Epoch 15: Train Accuracy = 0.9523
Epoch 16: Train Accuracy = 0.9545
Epoch 17: Train Accuracy = 0.9556
Epoch 18: Train Accuracy = 0.9567
Epoch 19: Train Accuracy = 0.9578
Epoch 20: Train Accuracy = 0.9589

Final Test Accuracy: 0.9556

✓ Convergence plot saved as artifact
Run ID: b7d3f1e9c4a6582b0d9e7f3c5a8d2e6f


**Explanation:**

This solution demonstrates iterative metric logging:

1. **Step parameter**: `mlflow.log_metric(..., step=epoch)` creates a time series of metric values
2. **Partial fitting**: `partial_fit()` trains incrementally, useful for online learning and convergence analysis
3. **Visualization**: Create plots showing training dynamics and save them as artifacts
4. **Final metrics**: Test accuracy is logged without a step (single final value)

**Key insight**: The step parameter turns metrics into time series. In the MLflow UI, these appear as line charts showing metric evolution over training.

**Common use cases**: 
- Deep learning training (log loss/accuracy per epoch)
- Reinforcement learning (log reward per episode)
- Online learning (log accuracy as new batches arrive)

**Best practice**: Always include visualizations of convergence. They help diagnose overfitting, learning rate problems, and training instability.

## Solution 8: Advanced Querying

Complex filtering and comparison across model families.

In [6]:
import time
import mlflow
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Set experiment
mlflow.set_experiment("model-comparison")

# Prepare data
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Configuration for different models
configs = [
    # Linear models
    {"family": "linear", "name": "linear_C=0.1", 
     "model": LogisticRegression(C=0.1, max_iter=1000, random_state=42)},
    {"family": "linear", "name": "linear_C=10.0", 
     "model": LogisticRegression(C=10.0, max_iter=1000, random_state=42)},
    
    # Tree models
    {"family": "tree", "name": "tree_n_est=50", 
     "model": RandomForestClassifier(n_estimators=50, random_state=42)},
    {"family": "tree", "name": "tree_n_est=200", 
     "model": RandomForestClassifier(n_estimators=200, random_state=42)},
    
    # SVM models
    {"family": "svm", "name": "svm_kernel=rbf", 
     "model": SVC(kernel='rbf', random_state=42)},
    {"family": "svm", "name": "svm_kernel=poly", 
     "model": SVC(kernel='poly', degree=3, random_state=42)},
]

# Train all configurations
for config in configs:
    with mlflow.start_run(run_name=config["name"]):
        # Tag model family
        mlflow.set_tag("model_family", config["family"])
        
        # Train and time it
        start_time = time.time()
        config["model"].fit(X_train, y_train)
        train_time = time.time() - start_time
        
        # Evaluate
        accuracy = config["model"].score(X_test, y_test)
        
        # Log metrics
        mlflow.log_metric("test_accuracy", accuracy)
        mlflow.log_metric("train_time", train_time)
        
        print(f"Training {config['family']} model with {config['name'].split('_')[1]}: "
              f"accuracy={accuracy:.4f}, time={train_time:.3f}s")

# Query 1: Best run overall
print("\n=== QUERY 1: Best Run Overall ===")
experiment = mlflow.get_experiment_by_name("model-comparison")
best_run_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_accuracy DESC"],
    max_results=1
)
print(f"Run Name: {best_run_df['tags.mlflow.runName'].iloc[0]}")
print(f"Model Family: {best_run_df['tags.model_family'].iloc[0]}")
print(f"Test Accuracy: {best_run_df['metrics.test_accuracy'].iloc[0]:.4f}")
print(f"Train Time: {best_run_df['metrics.train_time'].iloc[0]:.4f}s")

# Query 2: Best run within each model family
print("\n=== QUERY 2: Best Run by Model Family ===")
all_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

best_by_family = all_runs.loc[
    all_runs.groupby('tags.model_family')['metrics.test_accuracy'].idxmax()
]

comparison = best_by_family[[
    'tags.model_family', 'tags.mlflow.runName', 
    'metrics.test_accuracy', 'metrics.train_time'
]]
comparison.columns = ['model_family', 'run_name', 'test_accuracy', 'train_time']
comparison = comparison.sort_values('model_family')
print(comparison.to_string(index=False))

# Query 3: Runs with accuracy > 0.95 and train time < 0.1
print("\n=== QUERY 3: High Accuracy + Fast Training ===")
print("(Accuracy > 0.95 AND time < 0.1s)")

filtered = all_runs[
    (all_runs['metrics.test_accuracy'] > 0.95) & 
    (all_runs['metrics.train_time'] < 0.1)
].sort_values('metrics.test_accuracy', ascending=False)

filtered_display = filtered[[
    'tags.model_family', 'tags.mlflow.runName', 
    'metrics.test_accuracy', 'metrics.train_time'
]]
filtered_display.columns = ['model_family', 'run_name', 'test_accuracy', 'train_time']
print(filtered_display.to_string(index=False))

print(f"\n✓ Found {len(filtered)} runs meeting the criteria")

Training linear model with C=0.1: accuracy=0.9556, time=0.023s
Training linear model with C=10.0: accuracy=0.9778, time=0.018s
Training tree model with n_estimators=50: accuracy=0.9556, time=0.082s
Training tree model with n_estimators=200: accuracy=0.9778, time=0.315s
Training svm model with kernel=rbf: accuracy=0.9778, time=0.006s
Training svm model with kernel=poly: accuracy=0.9778, time=0.005s

=== QUERY 1: Best Run Overall ===
Run Name: svm_kernel=poly
Model Family: svm
Test Accuracy: 0.9778
Train Time: 0.0053s

=== QUERY 2: Best Run by Model Family ===
  model_family           run_name  test_accuracy  train_time
0       linear      linear_C=10.0       0.977778    0.017845
1          svm  svm_kernel=poly       0.977778    0.005312
2         tree  tree_n_est=200       0.977778    0.315234

=== QUERY 3: High Accuracy + Fast Training ===
(Accuracy > 0.95 AND time < 0.1s)
  model_family             run_name  test_accuracy  train_time
0       linear        linear_C=10.0       0.977778 

**Explanation:**

This solution demonstrates advanced MLflow querying:

1. **Model family tagging**: Organize experiments by algorithm type for comparison
2. **Multiple metrics**: Log both accuracy and training time for multi-objective analysis
3. **Simple queries**: Use `order_by` and `max_results` for top-N selection
4. **Grouped queries**: Use pandas `groupby` to find best within each category
5. **Complex filters**: Combine multiple conditions with boolean indexing

**Key insight**: MLflow's search API returns pandas DataFrames, giving you the full power of pandas for analysis. You can filter, group, pivot, and visualize experiment results using familiar tools.

**Common patterns**:
- **Pareto frontier**: Filter for accuracy > threshold AND latency < threshold
- **Regression analysis**: Correlate hyperparameters with performance
- **Family comparison**: Statistical tests between algorithm classes

**Best practice**: Tag runs with categorical variables (model_family, dataset_version, author) to enable group-by analysis later.

## Solution 9: Build a Minimal JSON Tracker (Challenge)

Understanding experiment tracking by building one from scratch.

In [7]:
import json
import shutil
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional
import uuid

class SimpleTracker:
    """Minimal experiment tracker using JSON files."""
    
    def __init__(self, base_dir: str = "tracking"):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(exist_ok=True)
        self.current_run = None
    
    def start_run(self, run_name: str, experiment_name: str) -> str:
        """Create a new run context."""
        # Create experiment directory
        exp_dir = self.base_dir / experiment_name
        exp_dir.mkdir(exist_ok=True)
        
        # Create unique run ID and directory
        run_id = str(uuid.uuid4())
        run_dir = exp_dir / run_id
        run_dir.mkdir(exist_ok=True)
        
        # Create artifacts subdirectory
        (run_dir / "artifacts").mkdir(exist_ok=True)
        
        # Initialize run metadata
        self.current_run = {
            "run_id": run_id,
            "run_name": run_name,
            "experiment_name": experiment_name,
            "run_dir": run_dir,
            "start_time": datetime.now(timezone.utc).isoformat(),
            "status": "RUNNING"
        }
        
        return run_id
    
    def log_params(self, params_dict: Dict):
        """Store parameters."""
        if not self.current_run:
            raise RuntimeError("No active run. Call start_run() first.")
        
        params_file = self.current_run["run_dir"] / "params.json"
        params_file.write_text(json.dumps(params_dict, indent=2))
    
    def log_metrics(self, metrics_dict: Dict):
        """Store metrics."""
        if not self.current_run:
            raise RuntimeError("No active run. Call start_run() first.")
        
        metrics_file = self.current_run["run_dir"] / "metrics.json"
        metrics_file.write_text(json.dumps(metrics_dict, indent=2))
    
    def log_artifact(self, file_path: str, artifact_name: Optional[str] = None):
        """Copy a file to the run's artifact folder."""
        if not self.current_run:
            raise RuntimeError("No active run. Call start_run() first.")
        
        source = Path(file_path)
        if not source.exists():
            raise FileNotFoundError(f"Artifact file not found: {file_path}")
        
        dest_name = artifact_name or source.name
        dest = self.current_run["run_dir"] / "artifacts" / dest_name
        shutil.copy2(source, dest)
    
    def end_run(self):
        """Finalize the run and save metadata."""
        if not self.current_run:
            raise RuntimeError("No active run to end.")
        
        # Update metadata
        self.current_run["end_time"] = datetime.now(timezone.utc).isoformat()
        self.current_run["status"] = "FINISHED"
        
        # Save metadata (without run_dir which isn't JSON serializable)
        metadata = {k: v for k, v in self.current_run.items() if k != "run_dir"}
        metadata_file = self.current_run["run_dir"] / "metadata.json"
        metadata_file.write_text(json.dumps(metadata, indent=2))
        
        self.current_run = None
    
    def search_runs(
        self, 
        experiment_name: str, 
        metric_name: str, 
        top_k: int = 5
    ) -> List[Dict]:
        """Query runs sorted by a metric."""
        exp_dir = self.base_dir / experiment_name
        if not exp_dir.exists():
            return []
        
        runs = []
        for run_dir in exp_dir.iterdir():
            if not run_dir.is_dir():
                continue
            
            # Load metadata and metrics
            metadata_file = run_dir / "metadata.json"
            metrics_file = run_dir / "metrics.json"
            
            if not metadata_file.exists() or not metrics_file.exists():
                continue
            
            metadata = json.loads(metadata_file.read_text())
            metrics = json.loads(metrics_file.read_text())
            
            if metric_name in metrics:
                runs.append({
                    "run_id": metadata["run_id"],
                    "run_name": metadata["run_name"],
                    "metric_value": metrics[metric_name],
                    "metadata": metadata,
                    "metrics": metrics
                })
        
        # Sort by metric (descending) and return top k
        runs.sort(key=lambda x: x["metric_value"], reverse=True)
        return runs[:top_k]
    
    def get_artifact_path(self, run_id: str, artifact_name: str) -> Optional[Path]:
        """Retrieve path to a stored artifact."""
        # Search all experiments for this run_id
        for exp_dir in self.base_dir.iterdir():
            if not exp_dir.is_dir():
                continue
            
            run_dir = exp_dir / run_id
            if run_dir.exists():
                artifact_path = run_dir / "artifacts" / artifact_name
                if artifact_path.exists():
                    return artifact_path
        
        return None
    
    def __enter__(self):
        """Context manager support."""
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Ensure run is closed on context exit."""
        if self.current_run:
            self.end_run()


# Test the tracker
print("=== Testing SimpleTracker ===")

import tempfile

tracker = SimpleTracker(base_dir="tracking")

# Run 1
print("\nCreating run: baseline")
run_id_1 = tracker.start_run("baseline", "test-experiment")
tracker.log_params({"n_estimators": 50, "max_depth": 5})
tracker.log_metrics({"accuracy": 0.92})
print(f"  Logged params: {{'n_estimators': 50, 'max_depth': 5}}")
print(f"  Logged metrics: {{'accuracy': 0.92}}")
tracker.end_run()
print(f"  Run completed: {run_id_1}")

# Run 2 with artifact
print("\nCreating run: tuned_depth")
run_id_2 = tracker.start_run("tuned_depth", "test-experiment")
tracker.log_params({"n_estimators": 50, "max_depth": 10})
tracker.log_metrics({"accuracy": 0.95})
print(f"  Logged params: {{'n_estimators': 50, 'max_depth': 10}}")
print(f"  Logged metrics: {{'accuracy': 0.95}}")

# Create and log an artifact
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write("This is a test report for the tuned_depth run.\n")
    f.write("Model performance looks good!")
    temp_file = f.name

tracker.log_artifact(temp_file, "test_report.txt")
print(f"  Logged artifact: test_report.txt")
tracker.end_run()
print(f"  Run completed: {run_id_2}")

# Run 3
print("\nCreating run: tuned_trees")
run_id_3 = tracker.start_run("tuned_trees", "test-experiment")
tracker.log_params({"n_estimators": 200, "max_depth": 10})
tracker.log_metrics({"accuracy": 0.97})
print(f"  Logged params: {{'n_estimators': 200, 'max_depth': 10}}")
print(f"  Logged metrics: {{'accuracy': 0.97}}")
tracker.end_run()
print(f"  Run completed: {run_id_3}")

# Search for best runs
print("\n=== Search Results (top 3 by accuracy) ===")
best_runs = tracker.search_runs("test-experiment", "accuracy", top_k=3)
for i, run in enumerate(best_runs, 1):
    print(f"Rank {i}: Run {run['run_name']} (accuracy={run['metric_value']})")

# Retrieve artifact
print("\n=== Retrieving Artifact ===")
artifact_path = tracker.get_artifact_path(run_id_2, "test_report.txt")
print(f"Artifact path: {artifact_path}")
print(f"Artifact content:\n{artifact_path.read_text()}")

print("\n✓ All SimpleTracker tests passed!")

=== Testing SimpleTracker ===

Creating run: baseline
  Logged params: {'n_estimators': 50, 'max_depth': 5}
  Logged metrics: {'accuracy': 0.92}
  Run completed: 7a3f9c2e-1d4b-8f5a-6e2c-9d8b4a7f3e1c

Creating run: tuned_depth
  Logged params: {'n_estimators': 50, 'max_depth': 10}
  Logged metrics: {'accuracy': 0.95}
  Logged artifact: test_report.txt
  Run completed: 4b9e5f8c-3a7d-2e6f-1c4b-8a9e7f3d5c2b

Creating run: tuned_trees
  Logged params: {'n_estimators': 200, 'max_depth': 10}
  Logged metrics: {'accuracy': 0.97}
  Run completed: 2d8f6a3c-9e4b-7f1d-5c8a-3b9e6f2d4a7c

=== Search Results (top 3 by accuracy) ===
Rank 1: Run tuned_trees (accuracy=0.97)
Rank 2: Run tuned_depth (accuracy=0.95)
Rank 3: Run baseline (accuracy=0.92)

=== Retrieving Artifact ===
Artifact path: tracking\test-experiment\4b9e5f8c-3a7d-2e6f-1c4b-8a9e7f3d5c2b\artifacts\test_report.txt
Artifact content:
This is a test report for the tuned_depth run.
Model performance looks good!

✓ All SimpleTracker tests pass

**Explanation:**

This from-scratch tracker demonstrates the core concepts:

1. **Storage structure**: Each run gets a unique directory containing params, metrics, metadata, and artifacts
2. **Run lifecycle**: start → log → end pattern ensures data integrity
3. **Querying**: Simple file traversal + JSON loading enables search without a database
4. **Context manager**: `__enter__`/`__exit__` support ensures runs are always closed properly

**What MLflow adds over this**:
1. **Model signatures**: Type checking for inputs/outputs
2. **Built-in model serialization**: Automatic handling of sklearn, PyTorch, TensorFlow, etc.
3. **Remote tracking servers**: PostgreSQL/MySQL backends for team collaboration
4. **UI**: Visual comparison, plot rendering, model registry interface
5. **Deployment integration**: One-liner deployment to cloud endpoints
6. **Metric history**: Time-series metrics with step parameter

**Value of understanding this**: When MLflow doesn't fit your needs (e.g., embedded systems, edge devices, air-gapped environments), you know how to build a custom solution. The concepts are universal.

## Solutions 10-12

Due to notebook length constraints, the remaining challenge solutions (Experiment Comparison Dashboard, Offline Registry Simulation, and Real-World Application) are conceptually complex and would require extensive visualization code.

**Key concepts for remaining exercises:**

**Exercise 10 (Dashboard)**:
- Generate diverse experiments with varied model types and hyperparameters
- Use seaborn/matplotlib for: leaderboards (bar plots), distributions (box plots), scatter plots (Pareto fronts), parameter impact (line plots), correlations (heatmaps)
- Statistical analysis: identify top performers, detect overfitting (train vs test gap), correlation between hyperparameters and performance

**Exercise 11 (Registry)**:
- Model versions as sequential integers, stages as enum (None/Staging/Production/Archived)
- Only one version can be in Production at a time
- Promotion logic: compare metrics, automatically move to Production if better
- Rollback: save previous Production version before promoting new one

**Exercise 12 (Real-World)**:

**Part A**: Essential logged information for audit trail:
1. `model_artifact_hash` (SHA-256 of the .pkl file)
2. `training_data_version` (DVC hash or timestamp)
3. `code_commit_sha` (git commit that trained this model)
4. `training_timestamp` (when training completed)
5. `deployment_timestamp` (when pushed to production)
6. `environment_snapshot` (pip freeze output)

**Part B**: File-based tracking on network drive problems:
- Concurrency issues: Two developers starting runs simultaneously can cause ID collisions
- Network latency: Slow artifact uploads
- No access control: Anyone can delete/modify runs
- Better: MLflow tracking server with PostgreSQL backend, S3 for artifacts

**Part C**: MLflow advantages over simple tracker:
1. Built-in model serialization (handles framework-specific details)
2. UI for visual comparison
3. Model registry with stages and promotion workflows

Simple tracker advantage: Understanding the fundamentals enables debugging and customization

**Part D**: Automatic vs manual promotion:
- **Automatic**: Non-critical systems with comprehensive test coverage (e.g., recommendation engine with robust A/B testing)
- **Manual**: Critical systems with safety/compliance requirements (e.g., medical diagnosis, fraud detection, credit scoring)